# Fabric Architecture Review - Setup (deploy everything)

Import and run **this one notebook** to deploy the whole review into the current Fabric workspace. It uses the Fabric REST API to create:

- a **Lakehouse** for intermediate + final outputs,
- four **stage notebooks** - `01_Collect`, `02_Analyze`, `03_Gold`, `04_Report`,
- a **Data pipeline** that runs them in order (Collect -> Analyze -> Gold -> Report),
- a **Direct Lake semantic model** + **Power BI report** (*Fabric Arch Review - Governance*) over a gold layer of Delta tables.

Re-running it is **idempotent** - existing items are updated in place.

## Prerequisites (set up these yourself first)

1. **The workspace** - run this notebook inside the Fabric workspace you want the items deployed to. It must be on a Fabric/Premium capacity.

Everything else (Lakehouse, notebooks, pipeline) is created **for you** by this notebook. The review itself authenticates as the identity that runs the pipeline.

## Parameters

In [ ]:
# Parameters
# --- GitHub repo to clone (public — no auth needed) ---
GITHUB_REPO_URL = "https://github.com/biro98/fabric-architecture-review.git"
GITHUB_BRANCH = "main"
# Release pinning: blank = the tag matching VERSION (e.g. v2026.06.0); set to pin an older release.
RELEASE_TAG = ""

# --- Deploy targets (where the items land) ---
WORKSPACE_ID = ""                 # blank = this notebook's workspace
LAKEHOUSE_NAME = "fabric_arch_review_lh"
PIPELINE_NAME = "Fabric Arch Review Pipeline"
NOTEBOOK_PREFIX = "FabricArchReview"

# --- Governance report (Direct Lake) deployed alongside the pipeline ---
SEMANTIC_MODEL_NAME = "Fabric Arch Review - Governance"
REPORT_NAME = "Fabric Arch Review - Governance"
DEPLOY_GOLD_REPORT = "true"   # "false" to skip the model + report

# --- Optional service principal (unattended / scheduled runs) ---
# Set SP_CLIENT_ID to have setup create a Service Principal cloud connection
# (you paste the secret into it afterward). Blank = run as the notebook identity.
SP_CLIENT_ID = ""                 # SP app (client) id; blank = notebook identity
SP_CONNECTION_NAME = "sp-fabric-arch-review"

# Everything else — client/engagement/reviewer names, tenant + workspace scope,
# the capacity-metrics flag, VertiPaq depth, and all analysis thresholds — is
# exposed as a *pipeline parameter*: set it per-run in the Fabric Run dialog. The
# baked-in defaults live in the "Deploy ... pipeline" cell below.

## 1. Clone the repo (to read the stage-notebook sources)

In [ ]:
import os, sys, shutil, subprocess
WORK_ROOT = "/tmp/fabric-arch-review-setup"
REPO_DIR = os.path.join(WORK_ROOT, "repo")
os.makedirs(WORK_ROOT, exist_ok=True)
_url = GITHUB_REPO_URL

def _fresh_clone():
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--depth", "1", _url, REPO_DIR], check=True)

# An existing single-branch clone won't have an origin/<branch> ref for a *different*
# branch, so reset to FETCH_HEAD (the tip we just fetched) instead. Refresh the remote
# URL first in case the repo changed; fall back to a clean clone on any failure.
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    try:
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", _url], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", GITHUB_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
    except subprocess.CalledProcessError:
        _fresh_clone()
else:
    _fresh_clone()
try:
    with open(os.path.join(REPO_DIR, "VERSION"), encoding="utf-8-sig") as _vf:
        _branch_version = _vf.read().strip()
except Exception:
    _branch_version = ""
RELEASE_TAG = (RELEASE_TAG or "").strip()
GITHUB_REF = RELEASE_TAG or (("v" + _branch_version) if _branch_version else GITHUB_BRANCH)
def _checkout_ref(ref):
    try:
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", ref], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "--force", "FETCH_HEAD"], check=True)
        return True
    except subprocess.CalledProcessError:
        return False
if GITHUB_REF != GITHUB_BRANCH and _checkout_ref(GITHUB_REF):
    print("Deploying pinned release ref:", GITHUB_REF)
else:
    if GITHUB_REF != GITHUB_BRANCH:
        print("WARNING: tag '" + GITHUB_REF + "' not found - deploying '" + GITHUB_BRANCH + "' tip (unpinned).")
    GITHUB_REF = GITHUB_BRANCH
try:
    DEPLOY_SHA = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    DEPLOY_SHA = ""
del _url
print("Repo cloned at", REPO_DIR, "| ref", GITHUB_REF, "| sha", DEPLOY_SHA[:8])

## 2. Fabric REST helpers

In [ ]:
import os, json, base64, time, requests, notebookutils
BASE = "https://api.fabric.microsoft.com/v1"

def _H():
    return {"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi"),
            "Content-Type": "application/json"}

def _poll(resp):
    if resp.status_code in (200, 201):
        return resp.json() if resp.content else {}
    if resp.status_code == 202:
        loc = resp.headers.get("Location")
        while loc:
            r = requests.get(loc, headers=_H())
            j = r.json() if r.content else {}
            s = str(j.get("status", ""))
            if s in ("Succeeded", "Completed"):
                rr = requests.get(loc.rstrip("/") + "/result", headers=_H())
                return rr.json() if (rr.status_code == 200 and rr.content) else j
            if s in ("Failed", "Cancelled", "Canceled"):
                raise RuntimeError("LRO " + s + ": " + json.dumps(j))
            time.sleep(int(r.headers.get("Retry-After", "3")))
    if resp.status_code >= 400:
        raise RuntimeError("Fabric HTTP " + str(resp.status_code) + " " + (resp.request.method or "") + " " + str(resp.url) + " -> " + (resp.text or "")[:4000])
    resp.raise_for_status()
    return {}

def find_item(wid, name, itype):
    r = requests.get(BASE + "/workspaces/" + wid + "/items?type=" + itype, headers=_H())
    for it in (r.json().get("value") if r.status_code == 200 else []) or []:
        if it.get("displayName") == name:
            return it
    return None

def ensure_lakehouse(wid, name):
    it = find_item(wid, name, "Lakehouse")
    if it:
        return it["id"]
    res = _poll(requests.post(BASE + "/workspaces/" + wid + "/lakehouses", headers=_H(),
                              json={"displayName": name}))
    if res.get("id"):
        return res["id"]
    return find_item(wid, name, "Lakehouse")["id"]

def _upsert(wid, name, itype_path, part_path, fmt, content_obj):
    payload = base64.b64encode(json.dumps(content_obj).encode("utf-8")).decode("ascii")
    definition = {"parts": [{"path": part_path, "payload": payload, "payloadType": "InlineBase64"}]}
    if fmt:
        definition["format"] = fmt
    itype = "Notebook" if itype_path == "notebooks" else "DataPipeline"
    it = find_item(wid, name, itype)
    if it:
        _poll(requests.post(BASE + "/workspaces/" + wid + "/" + itype_path + "/" + it["id"] + "/updateDefinition",
                            headers=_H(), json={"definition": definition}))
        return it["id"]
    res = _poll(requests.post(BASE + "/workspaces/" + wid + "/" + itype_path, headers=_H(),
                              json={"displayName": name, "definition": definition}))
    if res.get("id"):
        return res["id"]
    return find_item(wid, name, itype)["id"]

def upsert_notebook(wid, name, nb):
    return _upsert(wid, name, "notebooks", "notebook-content.ipynb", "ipynb", nb)

def upsert_pipeline(wid, name, pl):
    return _upsert(wid, name, "dataPipelines", "pipeline-content.json", None, pl)

def get_json(path):
    r = requests.get(BASE + path, headers=_H())
    return r.json() if (r.status_code == 200 and r.content) else {}

def upsert_definition(wid, itype_path, itype, name, definition):
    it = find_item(wid, name, itype)
    if it:
        _poll(requests.post(BASE + "/workspaces/" + wid + "/" + itype_path + "/" + it["id"] + "/updateDefinition",
                            headers=_H(), json={"definition": definition}))
        return it["id"]
    res = _poll(requests.post(BASE + "/workspaces/" + wid + "/" + itype_path, headers=_H(),
                              json={"displayName": name, "definition": definition}))
    if res.get("id"):
        return res["id"]
    return find_item(wid, name, itype)["id"]

def ensure_sp_connection(name, client_id, tenant_id):
    # Idempotently create a shareable cloud connection holding a Service Principal.
    # Secret is intentionally blank: paste it once via Manage connections post-setup.
    r = requests.get(BASE + "/connections?$top=200", headers=_H())
    for c in (r.json().get("value", []) if r.status_code == 200 and r.content else []):
        if c.get("displayName") == name:
            return c["id"]
    body = {
        "connectivityType": "ShareableCloud", "displayName": name,
        "connectionDetails": {"type": "WebForPipeline", "creationMethod": "WebForPipeline.Contents",
                              "parameters": [{"dataType": "Text", "name": "url", "value": "https://api.fabric.microsoft.com"}]},
        "privacyLevel": "Organizational",
        "credentialDetails": {"singleSignOnType": "None", "connectionEncryption": "NotEncrypted",
                              "skipTestConnection": True,
                              "credentials": {"credentialType": "ServicePrincipal", "tenantId": tenant_id,
                                              "servicePrincipalClientId": client_id, "servicePrincipalSecret": ""}},
    }
    r = requests.post(BASE + "/connections", headers=_H(), json=body)
    if r.status_code in (200, 201) and r.content:
        return r.json().get("id")
    print("  connection create failed:", r.status_code, (r.text or "")[:500])
    return name + " (create manually)"

print("REST helpers ready.")

## 3. Deploy Lakehouse + stage notebooks + pipeline

In [ ]:
import json, os, notebookutils
ctx = notebookutils.runtime.context
wid = WORKSPACE_ID or ctx.get("currentWorkspaceId") or ctx.get("workspaceId")
print("Target workspace:", wid)

lhid = ensure_lakehouse(wid, LAKEHOUSE_NAME)
print("Lakehouse:", LAKEHOUSE_NAME, lhid)

def load_nb(rel):
    with open(os.path.join(REPO_DIR, rel), encoding="utf-8") as f:
        nb = json.load(f)
    dep = nb.setdefault("metadata", {}).setdefault("dependencies", {})
    dep["lakehouse"] = {
        "default_lakehouse": lhid,
        "default_lakehouse_name": LAKEHOUSE_NAME,
        "default_lakehouse_workspace_id": wid,
        "known_lakehouses": [{"id": lhid}],
    }
    return nb

collect_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_01_Collect", load_nb("fabric/notebooks/01_collect.ipynb"))
analyze_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_02_Analyze", load_nb("fabric/notebooks/02_analyze.ipynb"))
report_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_03_Report", load_nb("fabric/notebooks/03_report.ipynb"))
gold_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_04_Gold", load_nb("fabric/notebooks/04_gold.ipynb"))
print("Notebooks:", collect_id, analyze_id, report_id, gold_id)


# --- Engagement + analysis pipeline parameters (set per-run in the Fabric Run
# dialog). These are NOT setup variables; their baked-in defaults live here. The
# activities below read each one via @pipeline().parameters.* so every value is
# selectable and overridable per run / on scheduled triggers. ---
common_keys = {
    "GITHUB_REPO_URL": GITHUB_REPO_URL, "GITHUB_BRANCH": GITHUB_BRANCH, "GITHUB_REF": GITHUB_REF,
    "CLIENT_NAME": "Contoso",
    "ENGAGEMENT_NAME": "Fabric Architecture Review",
    "REVIEWER_NAME": "",
}
collect_keys = {
    "TENANT_ID": "",                      # blank = this notebook's home tenant
    "WORKSPACE_IDS": "",                  # restrict to specific workspace GUIDs (blank = tenant-wide)
    "CAPACITY_METRICS_APP_INSTALLED": "false",
    "VERTIPAQ_STATS_READ_DATA": "false",  # "true" adds exact column cardinality (aggregate COUNT DAX)
    "ACTIVITY_DAYS_LOG": "7",             # activity-log lookback window in days (1-30)
    # Optional standing/unattended service-principal. Blank = run as the notebook
    # identity (default). Set SP_CLIENT_ID, then create a Service Principal cloud
    # connection named SP_CONNECTION_NAME manually and own the Collect notebook with
    # it. Optionally point a scheduled run
    # at a Key Vault secret instead via SP_SECRET_KEYVAULT + SP_SECRET_NAME.
    "SP_CLIENT_ID": SP_CLIENT_ID,         # SP appId; blank = notebook identity
    "SP_CONNECTION_NAME": SP_CONNECTION_NAME,  # cloud connection that holds the SP secret
    "SP_SECRET_KEYVAULT": "",             # optional Key Vault name/URI holding the SP secret
    "SP_SECRET_NAME": "",                 # optional secret name (used only for scheduled runs)
}
# Analysis thresholds (only consumed by the Analyze stage). Blank = use the curated
# default from config/thresholds.yaml; a value here overrides it. The comment after
# each key shows the built-in default.
threshold_keys = {
    "ARCH_MONOLITH_THRESHOLD": "",        # 50   max items before a workspace is "monolithic"
    "ARCH_PIPELINE_STALE_DAYS": "",       # 30   days without a pipeline run = stale
    "PERF_STALE_DAYS": "",                # 30   days without a model refresh = stale
    "PERF_LONG_REFRESH_HOURS": "",        # 2.0  refresh duration (h) flagged as long
    "PERF_FAIL_RATIO_THRESHOLD": "",      # 0.2  refresh failure ratio flagged
    "PERF_MODEL_SIZE_WARN_MB": "",        # 2048 model size (MB) warn
    "PERF_MODEL_SIZE_CRITICAL_MB": "",    # 8192 model size (MB) critical
    "PERF_REFRESH_OVERLAP_MIN": "",       # 2    overlap (min) between refreshes flagged
    "PERF_THROTTLE_CRITICAL_PCT": "",     # 100  throttle % critical
    "PERF_THROTTLE_WARN_PCT": "",         # 70   throttle % warn
    "PERF_JOB_FAIL_RATIO": "",            # 0.2  job failure ratio flagged
    "PERF_JOB_LONG_HOURS": "",            # 1.0  job duration (h) flagged as long
    "GOV_SHARE_VOLUME_THRESHOLD": "",     # 100  per-item shares before "broadly shared"
    "SEC_BROAD_ACCESS_THRESHOLD": "",     # 10   workspace principals = broad access
    "SEC_GATEWAY_MIN_VERSION": "",        # ""   minimum acceptable gateway version (blank = no check)
    "COST_SMALL_WORKSPACE_THRESHOLD": "", # 5    min workspaces to justify a large SKU
}

# Expose every value as a pipeline-level parameter so it is selectable and
# overridable in the Fabric Run dialog and on scheduled triggers (the activities
# below read them via @pipeline().parameters.* - the same way RUN_ID resolves).
_pp = {**common_keys, **collect_keys, **threshold_keys}
pipeline_parameters = {k: {"type": "string", "defaultValue": str(v)} for k, v in _pp.items()}

common = {k: "@pipeline().parameters." + k for k in common_keys}
common["RUN_ID"] = "@pipeline().RunId"
collect_params = dict(common)
collect_params.update({k: "@pipeline().parameters." + k for k in collect_keys})
analyze_params = dict(common)
analyze_params.update({k: "@pipeline().parameters." + k for k in threshold_keys})

def _P(d):
    return {k: {"value": v, "type": "string"} for k, v in d.items()}

def activity(name, nbid, params, depends=None):
    return {
        "name": name, "type": "TridentNotebook", "dependsOn": depends or [],
        "policy": {"timeout": "0.12:00:00", "retry": 1, "retryIntervalInSeconds": 120,
                   "secureOutput": False, "secureInput": False},
        "typeProperties": {"notebookId": nbid, "workspaceId": wid, "parameters": _P(params)},
    }

pipe = {"properties": {"parameters": pipeline_parameters, "activities": [
    activity("01 Collect", collect_id, collect_params),
    activity("02 Analyze", analyze_id, analyze_params, [{"activity": "01 Collect", "dependencyConditions": ["Succeeded"]}]),
    activity("03 Report", report_id, common, [{"activity": "02 Analyze", "dependencyConditions": ["Succeeded"]}]),
    activity("04 Gold", gold_id, common, [{"activity": "03 Report", "dependencyConditions": ["Succeeded"]}]),
]}}

pid = upsert_pipeline(wid, PIPELINE_NAME, pipe)
print("Pipeline:", PIPELINE_NAME, pid)

# --- Optional: Service Principal cloud connection -----------------------------
# Create the connection MANUALLY (Fabric can't store a secretless SP connection):
# Manage connections and gateways -> New -> Cloud -> type a name (SP_CONNECTION_NAME),
# auth = Service principal, fill tenant/client/secret. Then schedule the Collect
# notebook with that SP as owner. SP_CLIENT_ID here is only a reminder of which app.
SP_CLIENT_ID = (collect_keys.get("SP_CLIENT_ID") or "").strip()
SP_CONNECTION_NAME = (collect_keys.get("SP_CONNECTION_NAME") or "sp-fabric-arch-review").strip()
if SP_CLIENT_ID:
    print("Service principal mode: create cloud connection '" + SP_CONNECTION_NAME +
          "' for app " + SP_CLIENT_ID + " (auth = Service principal), then own the Collect notebook with it.")

print("\nDeployed. Open the workspace and run '" + PIPELINE_NAME + "' (or schedule it).")
print("Outputs land in Lakehouse '" + LAKEHOUSE_NAME + "' under Files/fabric-arch-review/<run-id>/report.md")

## 4. Deploy the Direct Lake governance model + report


In [ ]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Drop cached copies so a re-run picks up freshly cloned code (avoids stale-import bugs)
for _m in [m for m in list(sys.modules) if m == 'reports' or m.startswith('reports.')]:
    del sys.modules[_m]

if str(DEPLOY_GOLD_REPORT).strip().lower() in ("1", "true", "yes", "y", "on"):
    from reports.powerbi.deploy import wait_for_sql_endpoint, model_definition, report_definition
    print("Resolving Lakehouse SQL endpoint (can take ~1 min on a fresh Lakehouse)...")
    sql_server, sql_db = wait_for_sql_endpoint(get_json, wid, lhid)
    print("  SQL endpoint:", sql_server, "| database:", sql_db)

    # Bootstrap empty gold Delta tables so the Direct Lake model + report deploy
    # (and validate column references) even before the pipeline has ever run.
    # mode("ignore") is idempotent: it never clobbers tables the pipeline already filled.
    from pyspark.sql import SparkSession
    from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                                   DoubleType, BooleanType, TimestampType)
    from reports.powerbi.schema import GOLD_TABLES
    _spark = SparkSession.builder.getOrCreate()
    _T = {"string": StringType(), "int64": LongType(), "double": DoubleType(),
          "boolean": BooleanType(), "dateTime": TimestampType()}
    _tbl_root = "abfss://" + wid + "@onelake.dfs.fabric.microsoft.com/" + lhid + "/Tables"
    print("Bootstrapping empty gold tables (skipped where the pipeline already wrote data)...")
    for _t in GOLD_TABLES:
        _sch = StructType([StructField(c.name, _T[c.kind], True) for c in _t.columns])
        _spark.createDataFrame([], _sch).write.format("delta").mode("ignore").save(_tbl_root + "/" + _t.name)
    print("  gold tables ready:", ", ".join(t.name for t in GOLD_TABLES))

    # Record the DEPLOYED FAR version (from the cloned VERSION file) into a small
    # meta_deployment Delta table. The 04_Gold stage reads it to build gold_release
    # and the report's version banner flags when a newer release exists. Overwrite
    # keeps a single current row; it survives pipeline runs and is refreshed only
    # when you re-run setup.ipynb (i.e. when you actually update). Lakehouse history
    # tables (gold_*) are never touched here.
    from datetime import datetime, timezone
    _far_ver = "unknown"
    try:
        with open(os.path.join(REPO_DIR, "VERSION"), encoding="utf-8-sig") as _vf:
            _far_ver = _vf.read().strip() or "unknown"
    except Exception as _e:
        print("  (could not read VERSION file:", _e, ")")
    _meta_sch = StructType([
        StructField("version", StringType(), True),
        StructField("deployed_at_utc", StringType(), True),
        StructField("repo_url", StringType(), True),
        StructField("branch", StringType(), True),
        StructField("git_ref", StringType(), True),
        StructField("git_sha", StringType(), True),
    ])
    _meta_row = [(_far_ver, datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
                  GITHUB_REPO_URL, GITHUB_BRANCH, GITHUB_REF, DEPLOY_SHA)]
    (_spark.createDataFrame(_meta_row, _meta_sch).write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true").save(_tbl_root + "/meta_deployment"))
    print("  meta_deployment recorded: FAR v" + _far_ver + " @ " + GITHUB_REF)
    mid = upsert_definition(wid, "semanticModels", "SemanticModel", SEMANTIC_MODEL_NAME,
                            model_definition(SEMANTIC_MODEL_NAME, sql_server, sql_db))
    print("Semantic model:", SEMANTIC_MODEL_NAME, mid)
    rid = upsert_definition(wid, "reports", "Report", REPORT_NAME, report_definition(mid))
    print("Report:", REPORT_NAME, rid)
    print("\nGovernance report deployed. It shows data after the pipeline runs once")
    print("(deploy bootstraps the tables empty; the 03 Gold stage fills them each run).")
else:
    print("DEPLOY_GOLD_REPORT is false -> skipped the governance model + report.")